In [5]:
import json
import random
from openai import OpenAI
import os
import re
import numpy as np
import base64
import time
import pandas as pd 
from tqdm import tqdm
from word2number import w2n

# Load dataset

In [6]:
def load_dataset(annotation_path, question_path):
    try:
        with open(annotation_path, 'r') as f:
            annotations = json.load(f)['annotations']

        with open(question_path, 'r') as f:
            questions = json.load(f)['questions']

        # Select the dataset where ‘overall_scores’ == 1.0
        filtered_annotations = [
            annotation for annotation in annotations
            if annotation.get('overall_scores', {}).get('question') == 1.0 and
               annotation.get('overall_scores', {}).get('answer') == 1.0
        ]

        # Create a mapping from question ID to answer
        question_id_to_answer = {
            annotation['id']: {
                'answer': annotation['answer'],
                'answer_type': annotation['answer_type']  
            }
            for annotation in filtered_annotations
        }

        # Create a mapping from question ID to answer type
        question_id_to_answer_type = {
            annotation['id']: {
                'answer': annotation['answer'],
                'answer_type': annotation.get('answer_type', 'other')
            }
            for annotation in filtered_annotations
        }

        filtered_questions = [
            question for question in questions 
            if question['id'] in question_id_to_answer
        ]

        return filtered_questions, filtered_annotations, question_id_to_answer, question_id_to_answer_type

    except Exception as e:
        print(f"Error loading dataset: {e}")
        return [], [], {}, {}

def get_dataset(questions, question_id_to_answer, fraction=0.001, seed=42):
    # TODO：Increase quantity
# def get_dataset(questions, question_id_to_answer, fraction=0.05, seed=42):
    try:
        random.seed(seed)
        sample_size = max(1, int(len(questions) * fraction))
        sampled_questions = random.sample(questions, sample_size)
        sampled_truth_answers = [
            question_id_to_answer[q['id']]['answer'] 
            for q in sampled_questions
        ]
        
        for q in sampled_questions:
            q['answer_type'] = question_id_to_answer[q['id']]['answer_type']

        return sampled_questions, sampled_truth_answers

    except Exception as e:
        print(f"Error sampling dataset: {e}")
        return [], []

# Use first dataset
# def get_dataset(questions, annotations, fraction=0.05):
#     sample_size = int(len(questions) * fraction)
#     sampled_questions = questions[:sample_size]
#     sampled_annotations = annotations[:sample_size]
#     return sampled_questions, sampled_annotations

def encode_image(image_path):
    try:
        if not os.path.exists(image_path):
            print(f"Error: The image file at {image_path} was not found.")
            return None

        with open(image_path, "rb") as image_file:
            return base64.b64encode(image_file.read()).decode('utf-8')

    except Exception as e:
        print(f"An error occurred while encoding the image: {e}")
        return None

def parse_answer(input_str):
    if input_str is None:
        return None

    try:
        input_str = str(input_str).lower().strip()
        words = input_str.split()
        for i in range(len(words)):
            for j in range(i + 1, len(words) + 1):
                substring = ' '.join(words[i:j])
                try:
                    return str(w2n.word_to_num(substring))
                except:
                    continue

        matches = re.findall(r'\d+', input_str)
        if matches:
            return matches[-1]

        if "yes" in input_str:
            return "yes"
        elif "no" in input_str:
            return "no"

        return input_str

    except Exception as e:
        print(f"Error parsing answer '{input_str}': {e}")
        return input_str

# Single agent prediction

In [7]:
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def get_predict(question, image_base64, max_retries=3, retry_delay=2):
    if image_base64 is None:
        return None

    prompt = f"Question: {question}\nProvide an answer based on the image:"

    for attempt in range(max_retries):
        try:
            completion = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {
                        "role": "user",
                        "content": [
                            {"type": "text", "text": prompt},
                            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}},
                        ],
                    }
                ],
                max_tokens=100,
                temperature=0.3,
            )

            return completion.choices[0].message.content.strip()

        except Exception:
            time.sleep(retry_delay)
            continue

    return None


# Calculate accuracy

In [8]:
def compute_accuracy(question, truth_answer, predicted_answer, answer_type, max_retries=2, retry_delay=2):
    """Use GPT to evaluate the accuracy of the answer"""
    if predicted_answer is None:
        return 0
    
    # Construct evaluation prompt
    prompt = f"""
Evaluate if the following predicted answer is correct:

Question: {question}
True answer: {truth_answer}
Predicted answer: {predicted_answer}
Answer type: {answer_type}

Rules:
1. For yes/no type questions, determine if the predicted answer conveys the same meaning as the true answer.
2. For number type questions, check if the predicted answer contains the same number as the true answer.
3. For other type questions, determine if the predicted answer contains the key information from the true answer.
4. The predicted answer may be more detailed, but it's correct if it contains the right information.

Rate how well the predicted answer matches the correct answer on a scale of 0 to 1:
- 1.0: Perfect match or completely correct meaning
- 0.75: Mostly correct with minor differences
- 0.5: Partially correct
- 0.25: Slightly correct but missing key points
- 0.0: Completely incorrect or unrelated

Return only the numeric score, no explanation.
"""
    
    for attempt in range(max_retries):
        try:
            completion = client.chat.completions.create(
                model="gpt-4o-mini", 
                messages=[{"role": "user", "content": prompt}],
                max_tokens=10,
                temperature=0.3
            )
            
            return float(completion.choices[0].message.content.strip())
                
        except Exception:
            time.sleep(retry_delay)
            continue
    
    return None

# Load paths
annotation_path = "/Users/wt/PythonProjects/MultimodalComicAgent/dataset/simpsons/v1_Annotation_Val_simpsons_vqa.json"
question_path = "/Users/wt/PythonProjects/MultimodalComicAgent/dataset/simpsons/v1_Question_Val_simpsons_vqa.json"
images_dir = "/Users/wt/PythonProjects/MultimodalComicAgent/dataset/simpsons/val_images"

try:
    # Load dataset
    questions, annotations, question_id_to_answer, question_id_to_answer_type = load_dataset(annotation_path, question_path)

    if not questions:
        print("The 'questions' list is empty or not a list.")
        raise ValueError("Questions list is empty")

    # Get sample data TODO
    # sampled_questions, sampled_truth_answers = get_dataset(questions, question_id_to_answer, fraction=0.05)
    sampled_questions, sampled_truth_answers = get_dataset(questions, question_id_to_answer, fraction=0.001)

    if not sampled_questions:
        print("Failed to sample questions or empty sample")
        raise ValueError("No sampled questions")

    # Initialize results storage
    accuracies = []
    evaluation_results = []
    # Create an empty collection to store the processed problem IDs
    processed_question_ids = set() 

    # Process each question
    for question, truth_answer in tqdm(zip(sampled_questions, sampled_truth_answers),
                                     total=len(sampled_questions)):
        try:
            question_id = question['id']
            # Skip if already processed this question
            if question_id in processed_question_ids:
                continue
                
            processed_question_ids.add(question_id)
            
            question_text = question['question']
            image_relative_path = question['img_path']
            answer_type = question_id_to_answer_type[question_id]['answer_type']

            image_path = os.path.join(images_dir, image_relative_path)
            image_base64 = encode_image(image_path)

            if image_base64 is None:
                print(f"Skipping question ID {question_id} due to image encoding failure")
                continue

            model_answer = get_predict(question_text, image_base64)
            pred_solutions = [model_answer] if model_answer is not None else []

            if not pred_solutions:
                print(f"No prediction obtained for question ID {question_id}")
                continue

            accuracy = compute_accuracy(
                question=question_text,
                truth_answer=truth_answer,
                predicted_answer=model_answer,
                answer_type=answer_type
            )

            # Print results
            print(f"Question ID: {question_id}")
            print(f"Question: {question_text}")
            print(f"Answer Type: {answer_type}") 
            print(f"Truth Answer: {truth_answer}")
            if model_answer is not None:
                print(f"Predicted Answer: {model_answer}")
            if accuracy is not None:
                accuracies.append(accuracy)
                print(f"Accuracy: {accuracy:.4f}")
            else:
                print(f"Warning: No accuracy for question: {question_text}")

            # Store result
            result = {
                'question_id': question_id,
                'question': question_text,
                'answer_type': answer_type,
                'truth_answer': truth_answer,
                'predicted_answer': model_answer,
                'accuracy': accuracy
            }
            evaluation_results.append(result)
            accuracies.append(accuracy)

        except Exception as e:
            print(f"Error processing question {question.get('id', 'unknown')}: {e}")
            continue

    # Calculate average accuracy
    average_accuracy = np.mean(accuracies) if accuracies else 0
    print(f"Average Accuracy: {average_accuracy:.4f}")

except Exception as e:
    print(f"Unexpected error: {e}")
    average_accuracy = 0

 14%|█▍        | 1/7 [00:05<00:32,  5.47s/it]

Question ID: 77311
Question: what is on the shelf?
Answer Type: other
Truth Answer: book
Predicted Answer: I'm unable to see the specific details of the shelf in the image. If you describe what's on the shelf, I can help answer questions or provide more information!
Accuracy: 0.0000


 29%|██▊       | 2/7 [00:07<00:17,  3.43s/it]

Question ID: 12809
Question: how many people are in the picture?
Answer Type: number
Truth Answer: 1
Predicted Answer: There is one person in the picture.
Accuracy: 1.0000


 43%|████▎     | 3/7 [00:12<00:16,  4.02s/it]

...

Question ID: 36705
Question: what are the buildings made of?
Answer Type: other
Truth Answer: brick
Predicted Answer: The buildings in the image appear to be made of brick, as indicated by the red brick wall in the background. Additionally, there are elements like metal pipes and possibly concrete or other materials visible in the scene.
Accuracy: 1.0000


 86%|████████▌ | 6/7 [00:24<00:03,  3.90s/it]

Question ID: 33098
Question: is there a toy in the picture?
Answer Type: yes/no
Truth Answer: yes
Predicted Answer: Yes, there are toys in the picture, specifically LEGO bricks.
Accuracy: 1.0000


100%|██████████| 7/7 [00:31<00:00,  4.52s/it]

Question ID: 30161
Question: is there a man on a chair?
Answer Type: yes/no
Truth Answer: yes
Predicted Answer: Yes, there is a man sitting at a piano in the image.
Accuracy: 0.0000
Average Accuracy: 0.6786


# Save results

In [10]:
# Save results with explicit file handling to ensure overwriting works
evaluation_results = [r for r in evaluation_results if r['question_id'] != 'Average']
# Ensures no duplicate summary rows when saving results
unique_questions = len(set(r['question_id'] for r in evaluation_results))

# Add average accuracy as the last row
average_result = {
    'question_id': 'Average',
    'question': f'Total Questions: {unique_questions}',  
    'answer_type': 'All',  
    'truth_answer': '',
    'predicted_answer': '',
    'accuracy': average_accuracy 
}
evaluation_results.append(average_result)

column_order = [
    'question_id',
    'question',
    'answer_type',
    'truth_answer',
    'predicted_answer',
    'accuracy'
]

# Save to CSV
results_dir = "/Users/wt/PythonProjects/MultimodalComicAgent/results"
os.makedirs(results_dir, exist_ok=True)
output_path = os.path.join(results_dir, 'simpsons_evaluation_results_single_agent.csv')

# First, check if the file exists and explicitly remove it
if os.path.exists(output_path):
    try:
        os.remove(output_path)
        print(f"Existing file removed: {output_path}")
    except Exception as e:
        print(f"Error removing existing file: {e}")

# Convert to DataFrame and save with error handling
try:
    results_df = pd.DataFrame(evaluation_results)
    results_df = results_df[column_order]
    
    # Save with explicit file opening to ensure it closes properly
    results_df.to_csv(output_path, index=False)
    
    # Verify the file was created
    if os.path.exists(output_path):
        print(f"Results successfully saved to: {output_path}")
    else:
        print(f"Warning: File was not created at {output_path}")
except Exception as e:
    print(f"Error saving results to CSV: {e}")

Results successfully saved to: /Users/wt/PythonProjects/MultimodalComicAgent/results/simpsons_evaluation_results_single_agent.csv
